## AMY NJENGA 

### Explicit vs Implicit signals

In retail, signals such as a like, rating or comment are examples of explicit feedback on the user's experience.
Signals such as a view, an open or a click are more ambiguous. They are implicit, meaning it is difficult to gauge using that alone whether the user had a positive experience.

No interaction could probably mean the user had no opportunity to interact with the service, this is a safer assumption than dislike or lack of awareness

### Import Libraries

In [2]:
#%pip install implicit pandas numpy scipy 
import numpy as np 
import pandas as pd 
from scipy.sparse import csr_matrix 
from implicit.als import AlternatingLeastSquares 
np.random.seed(4060)

c:\Users\jkaru\dsa4060-week9-lab\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
users = [f"U{i:02d}" for i in range(1, 13)] 
items = [f"Track_{i:02d}" for i in range(1, 19)] 
event_types = ["play", "repeat_play", "save"] 
rows = [] 
for user in users: 
    preferred = np.random.choice(items, size=6, replace=False) 
    for item in preferred: 
        n = np.random.randint(1, 5) 
        for _ in range(n): 
            rows.append({ 
                "user_id": user, 
                "item_id": item, 
                "event_type": np.random.choice(event_types, 
                    p=[0.60, 0.25, 0.15]) 
        }) 

events = pd.DataFrame(rows) 
events.head(), events.shape

(  user_id   item_id   event_type
 0     U01  Track_02  repeat_play
 1     U01  Track_02  repeat_play
 2     U01  Track_02         play
 3     U01  Track_02         save
 4     U01  Track_06  repeat_play,
 (194, 3))

In [4]:
events['event_type'].value_counts() 
events.groupby('user_id').size().describe()

count    12.000000
mean     16.166667
std       3.511885
min      12.000000
25%      13.750000
50%      15.000000
75%      17.750000
max      23.000000
dtype: float64

### Checkpoint 1

We can infer from a repeat play or a save that the user may have some interest in the track, enough to want to experience it again.

We cannot infer how much the user enjoyed the track and what aspects of the track the user enjoyed or even whether the repeat plays were intentional or not.

A synthetic log may equally disperse the instances of the event types which differents from  real platform log which may have an overwhelming amount of one event log such as a play and a lower number of saves.



In [5]:
event_weight = { 
    "play": 1.0, 
    "repeat_play": 3.0, 
    "save": 5.0 
}

events['weight'] = events['event_type'].map(event_weight) 

interactions = (
    events
    .groupby(["user_id", "item_id"], as_index=False)
    .agg(strength=("weight", "sum"))
)


interactions.sort_values('strength', ascending=False).head(10)

,user_id,item_id,strength
1,U01,Track_02,12.0
71,U12,Track_12,12.0
61,U11,Track_03,12.0
69,U12,Track_09,12.0
14,U03,Track_11,11.0
70,U12,Track_10,11.0
8,U02,Track_10,10.0
11,U02,Track_17,10.0
67,U12,Track_05,10.0
54,U10,Track_06,10.0


I agree that one save should be worth 5 plays, this is because playing a song is not a strong indicator of preference or interest, a click could occur through accidents, autoplay or repeated exposure. This then makes a large number of clicks seem more valuable than they actually are. 

In terms of changes to the weights, I would give a save a higher weight, possibly a 7 with the rest remaining the same. This is because saving a trac seems like a strong indicator of the user's tastes.

In [6]:
alpha = 20
interactions['preference'] = (interactions['strength'] > 0).astype(int)
interactions['confidence'] = 1 + alpha * interactions['strength']

interactions[['user_id', 'item_id', 'strength',
'preference', 'confidence']].head()

,user_id,item_id,strength,preference,confidence
0,U01,Track_01,2.0,1,41.0
1,U01,Track_02,12.0,1,241.0
2,U01,Track_06,6.0,1,121.0
3,U01,Track_08,3.0,1,61.0
4,U01,Track_11,8.0,1,161.0


For a strength = 1, the preference would be 1 indicating that an interation has been observed, the confidence would be 21 which is slightly low. For a strength = 12, the preference is a binary score so it would be reduced to 1 and the confidence would be pushed to 241, which indicates a greater amunt of belief in the strength observed by the interaction


Unobserved pairs should be treated with low confidence because the true preference of the user would be unknown, treating the lack of an observation as a dislike risks affecting the model negatively, it is possible that if given the opportunity, the user would indicate a liking for the track

if the alpha is too large, it risks inflating the confidence we have in the interactions, making them seem more valuable to the model than they actually are

In [8]:
user_ids = sorted(interactions['user_id'].unique())
item_ids = sorted(interactions['item_id'].unique())
user_to_idx = {u: i for i, u in enumerate(user_ids)}
item_to_idx = {it: i for i, it in enumerate(item_ids)}
idx_to_item = {i: it for it, i in item_to_idx.items()}
row = interactions['user_id'].map(user_to_idx)
col = interactions['item_id'].map(item_to_idx)
data = interactions['strength'].astype(float)
user_item = csr_matrix(
(data, (row, col)),
shape=(len(user_ids), len(item_ids))
)
print('shape:', user_item.shape)
print('observed interactions:', user_item.nnz)
print('density:', user_item.nnz / (user_item.shape[0] *
user_item.shape[1]))
#print(user_item.head())

shape: (12, 18)
observed interactions: 72
density: 0.3333333333333333


This dataframe shows that one row is a user interaction and one column represents every item interaction mapped to that user.

A sparse matrix is more apprpriate because it conveys the true nature of the interactions, rather than a dense matrix that infers that a user hasinteracted with every single item

Density shows us that recommendation from a matrix is ideal and the problem requires the calculatin of a recommendation score.

In [ ]:
model = AlternatingLeastSquares(
factors=16,
regularization=0.10,
alpha=alpha,
iterations=25,
random_state=4060
)
model.fit(user_item)
print(model.user_factors.shape, model.item_factors.shape)

c:\Users\jkaru\dsa4060-week9-lab\.venv\Lib\site-packages\implicit\cpu\als.py:96: RuntimeWarning: OpenBLAS is configured to use 12 threads. It is highly recommended to disable its internal threadpool by setting the environment variable 'OPENBLAS_NUM_THREADS=1' or by calling 'threadpoolctl.threadpool_limits(1, "blas")'. Having OpenBLAS use a threadpool can lead to severe performance issues here.
  check_blas_config()
100%|██████████| 25/25 [00:00<00:00, 2518.32it/s]

(12, 16) (18, 16)


Factors = 16 controls the number of latent features used to represent users and items.
Regularization helps prevent overfitting by discouraging excessively large factor values.
A random seed makes results more reproducible 

In [ ]:
target_user = 'U01'
u = user_to_idx[target_user]
item_idx, scores = model.recommend(
u,
user_item[u],
N=5,
filter_already_liked_items=True
)
recommendations = pd.DataFrame({
'item_id': [idx_to_item[i] for i in item_idx],
'score': scores
})
seen = interactions.loc[
interactions['user_id'].eq(target_user),
['item_id', 'strength']
].sort_values('strength', ascending=False)
print('Observed history')
display(seen.head(10))
print('Recommended next')
display(recommendations)

Observed history


,item_id,strength
1,Track_02,12.0
4,Track_11,8.0
5,Track_13,6.0
2,Track_06,6.0
3,Track_08,3.0
0,Track_01,2.0


Recommended next


,item_id,score
0,Track_05,0.216540
1,Track_04,0.131311
2,Track_12,0.096400
3,Track_18,0.076339
4,Track_07,0.062073


Recommendations are plausible because they are detrmined from the strength of interactions for each score
Model scores are relative ranking scores, not probabilities, because they are not created to represent the likelihood of an item being selected.
Reranking may consider diversity of the users, freshness of the interactions, availability of information of the user and safety.

In [9]:
model = AlternatingLeastSquares(
factors=4,
regularization=0.10,
alpha=5,
iterations=25,
random_state=4060
)
model.fit(user_item)
print(model.user_factors.shape, model.item_factors.shape)

c:\Users\jkaru\dsa4060-week9-lab\.venv\Lib\site-packages\implicit\cpu\als.py:96: RuntimeWarning: OpenBLAS is configured to use 12 threads. It is highly recommended to disable its internal threadpool by setting the environment variable 'OPENBLAS_NUM_THREADS=1' or by calling 'threadpoolctl.threadpool_limits(1, "blas")'. Having OpenBLAS use a threadpool can lead to severe performance issues here.
  check_blas_config()
100%|██████████| 25/25 [00:00<00:00, 1762.70it/s]

(12, 4) (18, 4)


In [10]:
target_user = 'U01'
u = user_to_idx[target_user]
item_idx, scores = model.recommend(
u,
user_item[u],
N=5,
filter_already_liked_items=True
)
recommendations = pd.DataFrame({
'item_id': [idx_to_item[i] for i in item_idx],
'score': scores
})
seen = interactions.loc[
interactions['user_id'].eq(target_user),
['item_id', 'strength']
].sort_values('strength', ascending=False)
print('Observed history')
display(seen.head(10))
print('Recommended next')
display(recommendations)

Observed history


,item_id,strength
1,Track_02,12.0
4,Track_11,8.0
5,Track_13,6.0
2,Track_06,6.0
3,Track_08,3.0
0,Track_01,2.0


Recommended next


,item_id,score
0,Track_18,1.243253
1,Track_15,0.915596
2,Track_04,0.911362
3,Track_09,0.849095
4,Track_07,0.819938


In [11]:
model = AlternatingLeastSquares(
factors=32,
regularization=0.10,
alpha=40,
iterations=25,
random_state=4060
)
model.fit(user_item)
print(model.user_factors.shape, model.item_factors.shape)

100%|██████████| 25/25 [00:00<00:00, 827.13it/s]

(12, 32) (18, 32)


In [17]:
target_user = 'U01'
u = user_to_idx[target_user]
item_idx, scores = model.recommend(
u,
user_item[u],
N=5,
filter_already_liked_items=True
)

Experiment_recommendations = pd.DataFrame({
'item_id': [idx_to_item[i] for i in item_idx],
'score': scores
})
seen = interactions.loc[
interactions['user_id'].eq(target_user),
['item_id', 'strength']
].sort_values('strength', ascending=False)
print('Observed history')
display(seen.head(10))
print('Recommended next')
display(Experiment_recommendations)

Observed history


,item_id,strength
1,Track_02,12.0
4,Track_11,8.0
5,Track_13,6.0
2,Track_06,6.0
3,Track_08,3.0
0,Track_01,2.0


Recommended next


,item_id,score
0,Track_09,0.538252
1,Track_12,0.248889
2,Track_07,0.166217
3,Track_18,0.130227
4,Track_17,0.065186


In [18]:
# Copy the training cell, change ONE parameter, retrain,
# then rerun the recommendation cell.
comparison = pd.DataFrame({
'baseline_item': recommendations['item_id'],
'experiment_item': Experiment_recommendations['item_id']
})
comparison

,baseline_item,experiment_item
0,Track_09,Track_09
1,Track_12,Track_12
2,Track_07,Track_07
3,Track_18,Track_18
4,Track_17,Track_17


In [19]:
experiment_model = AlternatingLeastSquares(
    factors=16,
    regularization=0.10,
    alpha=40,
    iterations=25,
    random_state=4060
)

experiment_model.fit(user_item)

experiment_idx, experiment_scores = experiment_model.recommend(
    u,
    user_item[u],
    N=5,
    filter_already_liked_items=True
)

experiment_recommendations = pd.DataFrame({
    'item_id': [idx_to_item[i] for i in experiment_idx],
    'score': experiment_scores
})

comparison = pd.DataFrame({
    'baseline_item': recommendations['item_id'],
    'experiment_item': experiment_recommendations['item_id']
})

comparison

100%|██████████| 25/25 [00:00<00:00, 5051.67it/s]


,baseline_item,experiment_item
0,Track_09,Track_12
1,Track_12,Track_18
2,Track_07,Track_07
3,Track_18,Track_09
4,Track_17,Track_04


A random row split can place future interactions in the training set, causing data leakage.
Recall@K is more important when finding as many relevant items as possible matters more than limiting irrelevant recommendations.
Good offline results do not prove that users will enjoy recommendations in real situations. User testing is also necessary.

1. One assumption made when converting events into weights is that different user actions represent different levels of interest. A play was assigned a weight of 1, a repeat play 3, and a save 5. This assumes that saving a track indicates stronger interest than simply playing it.

2. Preference and confidence are different. Preference indicates whether a user interacted with an item, while confidence indicates how strongly the model should trust that interaction. For example, an interaction strength of 1 gives a preference of 1 and confidence of 21 when alpha is 20. A strength of 12 gives preference of 1 and confidence of 241.

3. Changing alpha from 20 to 40 increases the confidence assigned to observed interactions. This can influence the model's learned user and item factors and may change the Top-5 ranking.

4. One limitation of this model is that it uses synthetic data rather than real user behaviour. It also does not consider item descriptions or user demographics. A useful improvement would be to test the model using real interaction data with timestamps and evaluate its recommendations using Recall@5 and NDCG@5.